# ETF daily-grain quality checks
Read-only checks over captured sources. Set QBUSINESS_DATA_ROOT to the independent local etf-history directory.
Read README.md and etf-quality.json for the suspension, split, precision and historical-rule limitations. No research admission.

In [ ]:
from pathlib import Path
import hashlib, json, os
import pandas as pd
DATA_ROOT = Path(os.environ['QBUSINESS_DATA_ROOT'])
RECEIPTS = Path.cwd()  # directory containing this notebook and the receipts
manifest = json.loads((RECEIPTS / 'evidence-manifest.json').read_text())
for row in manifest:
    path = DATA_ROOT / row['path']
    assert hashlib.sha256(path.read_bytes()).hexdigest() == row['sha256'], row['path']
calendar = pd.read_csv(DATA_ROOT / 'validation/calendar.csv')
dates = pd.to_datetime(calendar.iloc[:, 0])
dates = pd.DatetimeIndex(dates[dates.between('2020-11-16', '2026-10-21')])
profile = []
for path in DATA_ROOT.glob('*/*/bars.parquet'):
    frame = pd.read_parquet(path)
    assert not frame.duplicated(['symbol', 'date']).any()
    profile.append({'source': str(path.relative_to(DATA_ROOT)), 'rows': len(frame),
                    'missing_dates': [str(d.date()) for d in dates.difference(frame.date)]})
pd.DataFrame(profile)
